# Load-Dependent Hardness GPR Example

This notebook builds a single-task Gaussian Process Regression workflow for experimental Vickers hardness data. It follows the paper emphasis that hardness is not only a composition-dependent property: the measured value also depends strongly on indentation load.

The workflow compares three experimental-only models:

- **Composition-only GPR**: a standard baseline using inorganic composition fingerprints.
- **Load-aware GPR**: the paper-aligned single-task model using composition fingerprints plus indentation load.
- **PI-GPR: load mean**: a physics-informed GPR whose prior mean captures the indentation-size trend and whose covariance learns the residual composition-dependent structure.

## Paper Reference

Mukherjee, M.; Ramprasad, R.; Sahu, H. **Load-dependent Hardness Prediction for Materials using Machine Learning.** Manuscript supplied with this example.

Key modeling points extracted from the manuscript and supporting information:

- Target: experimental Vickers hardness, reported in GPa.
- Dataset: 2,480 unique experimental records after duplicate removal.
- Essential measurement condition: applied indentation load, reported in N.
- Descriptor space: compositional, electronic, and periodic-table descriptors generated from inorganic formulas.
- Model family: Gaussian Process Regression with uncertainty estimates.
- First-stage scope here: single-task experimental model only; DFT-derived hardness values are reserved for the later multitask example.

## 1. Setup

The notebook is designed to run either from this example folder or from another working directory inside the repository.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

NOTEBOOK_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    """Find the local package root containing pyproject.toml and matgpr."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "matgpr").exists():
            return candidate
        sibling = candidate / "matgpr"
        if (sibling / "pyproject.toml").exists() and (sibling / "matgpr").exists():
            return sibling
    raise FileNotFoundError("Could not find the local matgpr package root.")


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Using package from: {PROJECT_ROOT}")

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
from gpytorch.utils.warnings import GPInputWarning
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

from matgpr import PhysicsInformedMean, featurize_compositions, fit_gpytorch_gpr

RANDOM_STATE = 42
plt.style.use("default")
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "legend.fontsize": 9,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)
torch.set_num_threads(1)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=GPInputWarning)

## 2. Load and Clean Experimental Data

The first hardness example uses only experimental records. A known spreadsheet artifact converts `FeB4` to `4-Feb`; this is corrected before formula parsing.

In [ ]:
DATA_PATH = NOTEBOOK_DIR / "dataset.csv"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "examples" / "hardness" / "dataset.csv"

raw_data = pd.read_csv(DATA_PATH, index_col=0)
print(f"Raw rows: {len(raw_data):,}")
raw_data.head()

In [ ]:
KNOWN_FORMULA_FIXES = {
    "4-Feb": "FeB4",
}

data = raw_data.copy()
data["composition"] = (
    data["composition"]
    .astype(str)
    .str.replace("\u00a0", "", regex=False)
    .str.strip()
    .replace(KNOWN_FORMULA_FIXES)
)
data["load_n"] = pd.to_numeric(data["load"], errors="coerce")
data["hardness_gpa"] = pd.to_numeric(data["hardness"], errors="coerce")

if "category" in data.columns:
    data = data[data["category"].astype(str).str.lower().eq("exp")].copy()

data = data.dropna(subset=["composition", "load_n", "hardness_gpa"])
data = data[data["load_n"] > 0].copy()
data = data.drop_duplicates(subset=["composition", "load_n", "hardness_gpa"]).reset_index(drop=True)
data["log_load_n"] = np.log1p(data["load_n"])

summary = pd.DataFrame(
    {
        "rows": [len(data)],
        "unique_compositions": [data["composition"].nunique()],
        "min_load_n": [data["load_n"].min()],
        "max_load_n": [data["load_n"].max()],
        "min_hardness_gpa": [data["hardness_gpa"].min()],
        "max_hardness_gpa": [data["hardness_gpa"].max()],
    }
)
summary

## 3. Inorganic Composition Fingerprints

The package featurizer creates 60 descriptors from 10 elemental properties and 6 composition statistics. The descriptor names intentionally follow the paper style where `fwm` is the fraction-weighted mean and `ad` is the fraction-weighted absolute deviation.

In [ ]:
fingerprint_result = featurize_compositions(data["composition"], errors="coerce")
if not fingerprint_result.failed.empty:
    display(fingerprint_result.failed)
    raise ValueError("Resolve formula parsing failures before modeling.")

composition_features = fingerprint_result.features.columns.tolist()
model_data = pd.concat([data.reset_index(drop=True), fingerprint_result.features], axis=1)

print(f"Composition descriptor count: {len(composition_features)}")
model_data[["composition", "load_n", "hardness_gpa", *composition_features[:6]]].head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.3))

axes[0].hist(model_data["hardness_gpa"], bins=35, color="#2f6f8f", alpha=0.9)
axes[0].set_xlabel("Experimental hardness (GPa)")
axes[0].set_ylabel("Count")
axes[0].set_title("Target distribution")

axes[1].hist(model_data["load_n"], bins=35, color="#8a5a44", alpha=0.9)
axes[1].set_xlabel("Applied load (N)")
axes[1].set_title("Load distribution")

axes[2].scatter(
    model_data["load_n"],
    model_data["hardness_gpa"],
    s=18,
    alpha=0.55,
    color="#4a7c59",
    edgecolor="none",
)
axes[2].set_xscale("log")
axes[2].set_xlabel("Applied load (N, log scale)")
axes[2].set_ylabel("Experimental hardness (GPa)")
axes[2].set_title("Load dependence")

fig.tight_layout()

## 4. Model Definitions

The physics-informed model uses a simple indentation-size-effect mean function:

`H(load) = H_floor + A / sqrt(load + P0)`

`H_floor`, `A`, and `P0` are learned jointly with the GP hyperparameters. The mean function captures the common trend that measured hardness decreases with increasing indentation load, while the GP covariance learns composition-dependent residuals.

In [ ]:
TARGET_COLUMN = "hardness_gpa"
LOAD_FEATURES = ["load_n", "log_load_n"]

MODEL_CONFIGS = {
    "composition_only": {
        "label": "Composition-only GPR",
        "feature_columns": composition_features,
        "physics_mean": False,
    },
    "load_aware": {
        "label": "Load-aware GPR",
        "feature_columns": composition_features + LOAD_FEATURES,
        "physics_mean": False,
    },
    "physics_load_mean": {
        "label": "PI-GPR: load mean",
        "feature_columns": composition_features + LOAD_FEATURES,
        "physics_mean": True,
    },
}
MODEL_ORDER = list(MODEL_CONFIGS)
MODEL_LABELS = {key: config["label"] for key, config in MODEL_CONFIGS.items()}

TRAINING_ITER = 150
PRODUCTION_TRAINING_ITER = 250
LEARNING_CURVE_REPEATS = 5
LEARNING_CURVE_PERCENTS = np.arange(10, 71, 10)
CV_SPLITS = 10

In [ ]:
def indentation_size_effect_mean(features, parameters):
    """Physics mean for load-dependent hardness in original units."""
    load = torch.clamp(features["load_n"], min=0.0)
    return parameters["hardness_floor"] + parameters["load_strength"] / torch.sqrt(
        load + parameters["load_offset"]
    )


def build_load_physics_mean(feature_columns, scaler, y_train):
    """Build the learnable load-dependent mean module after feature scaling."""
    feature_means = dict(zip(feature_columns, scaler.mean_))
    feature_stds = dict(zip(feature_columns, scaler.scale_))
    y_train = np.asarray(y_train, dtype=float)

    return PhysicsInformedMean(
        equation=indentation_size_effect_mean,
        feature_indices={"load_n": feature_columns.index("load_n")},
        learnable_parameters={
            "hardness_floor": float(np.percentile(y_train, 25)),
            "load_strength": float(max(np.percentile(y_train, 75) - np.percentile(y_train, 25), 1.0)),
            "load_offset": 1.0,
        },
        positive_parameters=("load_strength", "load_offset"),
        feature_means={"load_n": feature_means["load_n"]},
        feature_stds={"load_n": feature_stds["load_n"]},
    )


def fit_hardness_gpr(model_key, train_frame, *, training_iter=TRAINING_ITER):
    """Fit one configured GPR model and keep its scaler with the result."""
    config = MODEL_CONFIGS[model_key]
    feature_columns = config["feature_columns"]
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_frame[feature_columns])
    y_train = train_frame[TARGET_COLUMN].to_numpy(dtype=float)

    mean_module = None
    if config["physics_mean"]:
        mean_module = build_load_physics_mean(feature_columns, scaler, y_train)

    result = fit_gpytorch_gpr(
        X_train,
        y_train,
        kernel="matern",
        ard=True,
        mean_module=mean_module,
        lr=0.03,
        training_iter=training_iter,
        initial_noise=0.05,
        standardize_y=True,
        verbose=False,
    )
    return {
        "model_key": model_key,
        "label": config["label"],
        "feature_columns": feature_columns,
        "scaler": scaler,
        "result": result,
    }


def predict_hardness(fitted_model, frame, *, confidence_level=0.95):
    X = fitted_model["scaler"].transform(frame[fitted_model["feature_columns"]])
    return fitted_model["result"].predict(X, confidence_level=confidence_level)


def regression_summary(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    r_value = pearsonr(y_true, y_pred).statistic if len(np.unique(y_true)) > 1 else np.nan
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "r": float(r_value),
    }


def hardness_strata(frame, bins=8):
    return pd.qcut(frame[TARGET_COLUMN], q=bins, labels=False, duplicates="drop")


def stratified_subset(frame, n_rows, *, random_state):
    if n_rows >= len(frame):
        return frame.copy()
    strata = hardness_strata(frame)
    try:
        subset, _ = train_test_split(
            frame,
            train_size=n_rows,
            random_state=random_state,
            stratify=strata,
        )
        return subset.reset_index(drop=True)
    except ValueError:
        return frame.sample(n=n_rows, random_state=random_state).reset_index(drop=True)

## 5. External Test Split

A fixed 30 percent test set is held out before the learning-curve experiment. The learning-curve x-axis reports the percentage of the full dataset used for training.

In [ ]:
train_pool, external_test = train_test_split(
    model_data,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=hardness_strata(model_data),
)
train_pool = train_pool.reset_index(drop=True)
external_test = external_test.reset_index(drop=True)

print(f"Training pool rows: {len(train_pool):,}")
print(f"External test rows: {len(external_test):,}")

## 6. Learning Curve Experiment

Each point is repeated over random stratified subsets from the 70 percent training pool. Error bars in the plots show one standard deviation across repeats.

In [ ]:
learning_records = []

for training_percent in LEARNING_CURVE_PERCENTS:
    n_train = int(round(len(model_data) * training_percent / 100))
    for repeat in range(LEARNING_CURVE_REPEATS):
        seed = RANDOM_STATE + 1000 * int(training_percent) + repeat
        subset = stratified_subset(train_pool, n_train, random_state=seed)

        for model_key in MODEL_ORDER:
            fitted = fit_hardness_gpr(model_key, subset, training_iter=TRAINING_ITER)
            prediction = predict_hardness(fitted, external_test)
            metrics = regression_summary(external_test[TARGET_COLUMN], prediction.mean)
            learning_records.append(
                {
                    "training_percent": int(training_percent),
                    "repeat": repeat,
                    "n_train": len(subset),
                    "model_key": model_key,
                    "model": MODEL_LABELS[model_key],
                    **metrics,
                }
            )

learning_results = pd.DataFrame(learning_records)
learning_summary = (
    learning_results
    .groupby(["training_percent", "model_key", "model"], as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        r2_mean=("r2", "mean"),
        r2_std=("r2", "std"),
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
    )
)
learning_summary.head()

## 7. Learning Curve Plots

The central comparison is whether explicitly adding load and the load-dependent mean improves low-data and moderate-data performance relative to a composition-only GPR.

In [ ]:
MODEL_COLORS = {
    "composition_only": "#4c5a72",
    "load_aware": "#2f6f8f",
    "physics_load_mean": "#b85c38",
}

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.1), sharex=True)

for model_key in MODEL_ORDER:
    subset = learning_summary[learning_summary["model_key"] == model_key]
    label = MODEL_LABELS[model_key]
    color = MODEL_COLORS[model_key]
    axes[0].errorbar(
        subset["training_percent"],
        subset["rmse_mean"],
        yerr=subset["rmse_std"],
        marker="o",
        linewidth=2.0,
        capsize=3,
        label=label,
        color=color,
    )
    axes[1].errorbar(
        subset["training_percent"],
        subset["r2_mean"],
        yerr=subset["r2_std"],
        marker="o",
        linewidth=2.0,
        capsize=3,
        label=label,
        color=color,
    )

axes[0].set_ylabel("External test RMSE (GPa)")
axes[1].set_ylabel("External test R2")
for ax in axes:
    ax.set_xlabel("Training data (% of full dataset)")
    ax.set_xticks(LEARNING_CURVE_PERCENTS)
    ax.legend(frameon=False)

fig.tight_layout()

## 8. Select the Best Low-Data Model

Model selection is based on 20 percent training data because this highlights performance in the low-data regime while still using enough samples for stable GPR fitting.

In [ ]:
SELECTION_PERCENT = 20
selection_table = (
    learning_summary[learning_summary["training_percent"] == SELECTION_PERCENT]
    .sort_values(["rmse_mean", "rmse_std"], ascending=[True, True])
    .reset_index(drop=True)
)
best_model_key = selection_table.loc[0, "model_key"]
best_model_label = MODEL_LABELS[best_model_key]

print(f"Selected model: {best_model_label}")
selection_table[["model", "rmse_mean", "rmse_std", "r2_mean", "r2_std"]]

## 9. 90/10 Validation With 10-Fold Cross-Validation

Before fitting the production model, the selected model is evaluated using a conventional validation protocol: 90 percent train, 10 percent held-out test, and 10-fold cross-validation on the 90 percent training partition. This follows the paper's emphasis on repeated split/CV statistics while keeping the example single-task and experimental-only.

In [ ]:
validation_train, validation_test = train_test_split(
    model_data,
    test_size=0.10,
    random_state=RANDOM_STATE + 7,
    stratify=hardness_strata(model_data),
)
validation_train = validation_train.reset_index(drop=True)
validation_test = validation_test.reset_index(drop=True)

cv_records = []
cv_strata = hardness_strata(validation_train, bins=CV_SPLITS)
cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, val_idx) in enumerate(cv.split(validation_train, cv_strata), start=1):
    fold_train = validation_train.iloc[train_idx].reset_index(drop=True)
    fold_val = validation_train.iloc[val_idx].reset_index(drop=True)
    fitted = fit_hardness_gpr(best_model_key, fold_train, training_iter=TRAINING_ITER)

    train_pred = predict_hardness(fitted, fold_train)
    val_pred = predict_hardness(fitted, fold_val)
    train_metrics = regression_summary(fold_train[TARGET_COLUMN], train_pred.mean)
    val_metrics = regression_summary(fold_val[TARGET_COLUMN], val_pred.mean)

    cv_records.append({"fold": fold, "split": "CV train", **train_metrics})
    cv_records.append({"fold": fold, "split": "CV validation", **val_metrics})

cv_results = pd.DataFrame(cv_records)
cv_summary = (
    cv_results
    .groupby("split")
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        r2_mean=("r2", "mean"),
        r2_std=("r2", "std"),
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        r_mean=("r", "mean"),
        r_std=("r", "std"),
    )
    .reset_index()
)
cv_summary

In [ ]:
validation_model = fit_hardness_gpr(
    best_model_key,
    validation_train,
    training_iter=PRODUCTION_TRAINING_ITER,
)
validation_train_pred = predict_hardness(validation_model, validation_train)
validation_test_pred = predict_hardness(validation_model, validation_test)

validation_train_metrics = regression_summary(validation_train[TARGET_COLUMN], validation_train_pred.mean)
validation_test_metrics = regression_summary(validation_test[TARGET_COLUMN], validation_test_pred.mean)

validation_metrics = pd.DataFrame(
    [
        {"split": "Train", **validation_train_metrics},
        {"split": "Held-out test", **validation_test_metrics},
    ]
)
validation_metrics

In [ ]:
def format_metric(mean_value, std_value):
    return f"{mean_value:.3f} +/- {std_value:.3f}"

cv_table = cv_summary.copy()
for metric in ["rmse", "r2", "mae", "r"]:
    cv_table[metric.upper()] = [
        format_metric(row[f"{metric}_mean"], row[f"{metric}_std"]) for _, row in cv_table.iterrows()
    ]
cv_table = cv_table[["split", "RMSE", "R2", "MAE", "R"]]

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.9), gridspec_kw={"width_ratios": [0.92, 1.35]})

axes[0].axis("off")
axes[0].set_title(f"{best_model_label}: 10-fold CV", pad=10)
table = axes[0].table(
    cellText=cv_table.values,
    colLabels=cv_table.columns,
    cellLoc="center",
    colLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1.05, 1.45)
for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(0.45)
    if row == 0:
        cell.set_facecolor("#e9eef2")
        cell.set_text_props(weight="bold")

ax = axes[1]
for frame, pred, split, color in [
    (validation_train, validation_train_pred, "Train", "#2f6f8f"),
    (validation_test, validation_test_pred, "Held-out test", "#b85c38"),
]:
    ax.errorbar(
        frame[TARGET_COLUMN],
        pred.mean,
        yerr=pred.std,
        fmt="o",
        ms=4.0,
        alpha=0.65,
        elinewidth=0.7,
        capsize=1.8,
        color=color,
        label=split,
    )

limits = [
    min(validation_train[TARGET_COLUMN].min(), validation_test[TARGET_COLUMN].min()),
    max(validation_train[TARGET_COLUMN].max(), validation_test[TARGET_COLUMN].max()),
]
pad = 0.05 * (limits[1] - limits[0])
limits = [limits[0] - pad, limits[1] + pad]
ax.plot(limits, limits, color="#2b2b2b", linewidth=1.1, linestyle="--")
ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_xlabel("Experimental hardness (GPa)")
ax.set_ylabel("Predicted hardness (GPa)")
ax.set_title("90/10 parity with predictive std")
ax.legend(frameon=False)

text = (
    f"Train RMSE={validation_train_metrics['rmse']:.2f}, R2={validation_train_metrics['r2']:.2f}, "
    f"r={validation_train_metrics['r']:.2f}\\n"
    f"Test RMSE={validation_test_metrics['rmse']:.2f}, R2={validation_test_metrics['r2']:.2f}, "
    f"r={validation_test_metrics['r']:.2f}"
)
ax.text(0.04, 0.96, text, transform=ax.transAxes, va="top", ha="left", fontsize=9)

fig.tight_layout()

## 10. Descriptor Interpretation

A simple Pearson screen is useful before deeper interpretation. This mirrors the supporting information analysis by showing which descriptors have the strongest monotonic association with experimental hardness.

In [ ]:
best_feature_columns = MODEL_CONFIGS[best_model_key]["feature_columns"]
correlation_records = []
for column in best_feature_columns:
    values = model_data[column].to_numpy(dtype=float)
    if np.nanstd(values) == 0:
        continue
    r_value = pearsonr(values, model_data[TARGET_COLUMN]).statistic
    correlation_records.append({"feature": column, "r": r_value, "abs_r": abs(r_value)})

correlation_table = (
    pd.DataFrame(correlation_records)
    .sort_values("abs_r", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(7.0, 5.0))
colors = ["#b85c38" if value >= 0 else "#2f6f8f" for value in correlation_table["r"]]
ax.barh(correlation_table["feature"][::-1], correlation_table["r"][::-1], color=colors[::-1])
ax.axvline(0, color="#333333", linewidth=0.9)
ax.set_xlabel("Pearson r with experimental hardness")
ax.set_title("Top descriptor correlations")
fig.tight_layout()

correlation_table

## 11. Production Model on 100 Percent of Experimental Data

The production model uses the selected best model family and all experimental data. This is the model to serialize or reuse for prediction once the validation behavior is acceptable.

In [ ]:
production_model = fit_hardness_gpr(
    best_model_key,
    model_data,
    training_iter=PRODUCTION_TRAINING_ITER,
)
production_prediction = predict_hardness(production_model, model_data)
production_metrics = regression_summary(model_data[TARGET_COLUMN], production_prediction.mean)

print(f"Production model: {best_model_label}")
print(production_metrics)

if best_model_key == "physics_load_mean":
    print("Learned load-mean parameters:")
    print(production_model["result"].model.mean_module.current_parameter_values())

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 5.0))
ax.errorbar(
    model_data[TARGET_COLUMN],
    production_prediction.mean,
    yerr=production_prediction.std,
    fmt="o",
    ms=3.7,
    alpha=0.55,
    elinewidth=0.6,
    capsize=1.4,
    color="#2f6f8f",
)
limits = [model_data[TARGET_COLUMN].min(), model_data[TARGET_COLUMN].max()]
pad = 0.05 * (limits[1] - limits[0])
limits = [limits[0] - pad, limits[1] + pad]
ax.plot(limits, limits, color="#2b2b2b", linewidth=1.1, linestyle="--")
ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_xlabel("Experimental hardness (GPa)")
ax.set_ylabel("Predicted hardness (GPa)")
ax.set_title("Production-model fit")
ax.text(
    0.04,
    0.96,
    f"RMSE={production_metrics['rmse']:.2f} GPa\\nR2={production_metrics['r2']:.2f}\\nr={production_metrics['r']:.2f}",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=9,
)
fig.tight_layout()

## 12. SHAP Analysis for the Production Model

Permutation SHAP is run on the production model to identify which load and composition descriptors drive predicted hardness. For tractability and interpretability, the SHAP candidate set always includes the load features and then adds the most target-correlated composition descriptors.

In [ ]:
def select_shap_features(frame, feature_columns, target_column, *, always_include=(), max_features=30):
    selected = [feature for feature in always_include if feature in feature_columns]
    records = []
    for feature in feature_columns:
        values = pd.to_numeric(frame[feature], errors="coerce").to_numpy(dtype=float)
        target = frame[target_column].to_numpy(dtype=float)
        mask = np.isfinite(values) & np.isfinite(target)
        if mask.sum() < 3 or np.nanstd(values[mask]) == 0:
            score = 0.0
        else:
            score = abs(pearsonr(values[mask], target[mask]).statistic)
        records.append({"feature": feature, "score": score})
    ranked = pd.DataFrame(records).sort_values("score", ascending=False)
    for feature in ranked["feature"]:
        if feature not in selected:
            selected.append(feature)
        if len(selected) >= min(max_features, len(feature_columns)):
            break
    return selected, ranked


production_feature_columns = production_model["feature_columns"]
SHAP_MAX_FEATURES = min(30, len(production_feature_columns))
SHAP_BACKGROUND_SIZE = min(50, len(model_data))
SHAP_EXPLAIN_SIZE = min(80, len(model_data))
SHAP_FEATURES, shap_feature_ranking = select_shap_features(
    model_data,
    production_feature_columns,
    TARGET_COLUMN,
    always_include=LOAD_FEATURES,
    max_features=SHAP_MAX_FEATURES,
)

reference_values = model_data[production_feature_columns].median(numeric_only=True)
shap_background = shap.sample(model_data[SHAP_FEATURES], SHAP_BACKGROUND_SIZE, random_state=RANDOM_STATE)
shap_explain = shap.sample(model_data[SHAP_FEATURES], SHAP_EXPLAIN_SIZE, random_state=RANDOM_STATE + 1)


def production_predict_for_shap(shap_frame):
    shap_frame = pd.DataFrame(shap_frame, columns=SHAP_FEATURES)
    full_frame = pd.DataFrame(
        np.repeat(reference_values.to_numpy()[None, :], len(shap_frame), axis=0),
        columns=production_feature_columns,
    )
    full_frame.loc[:, SHAP_FEATURES] = shap_frame.to_numpy()
    X_scaled = production_model["scaler"].transform(full_frame[production_feature_columns])
    return production_model["result"].predict(X_scaled, return_std=False).mean


explainer = shap.PermutationExplainer(production_predict_for_shap, shap_background)
shap_values = explainer(
    shap_explain,
    max_evals=2 * len(SHAP_FEATURES) + 1,
    batch_size=16,
)
shap_value_array = np.asarray(shap_values.values)
if shap_value_array.ndim == 3:
    shap_value_array = shap_value_array[:, :, 0]

shap_importance = (
    pd.DataFrame(
        {
            "feature": SHAP_FEATURES,
            "mean_abs_shap": np.abs(shap_value_array).mean(axis=0),
        }
    )
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
shap_importance.head(10).round(4)

In [ ]:
top_features = shap_importance.head(min(10, len(shap_importance)))["feature"].tolist()
bar_data = shap_importance.head(min(10, len(shap_importance))).iloc[::-1]
shap_predictions = production_predict_for_shap(shap_explain)

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.5), gridspec_kw={"width_ratios": [1.05, 1.0]})
axes[0].barh(bar_data["feature"], bar_data["mean_abs_shap"], color="#2f6f8f")
axes[0].set_xlabel("Mean absolute SHAP value (GPa)")
axes[0].set_title("Top SHAP features")

feature = top_features[0]
feature_index = SHAP_FEATURES.index(feature)
scatter = axes[1].scatter(
    shap_explain[feature],
    shap_value_array[:, feature_index],
    c=shap_predictions,
    cmap="viridis",
    s=34,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.25,
)
axes[1].axhline(0, color="#333333", linewidth=0.9, linestyle="--")
axes[1].set_xlabel(feature)
axes[1].set_ylabel("SHAP value (GPa)")
axes[1].set_title(f"Dependence: {feature}")
colorbar = fig.colorbar(scatter, ax=axes[1])
colorbar.set_label("Predicted hardness")
fig.tight_layout()

## 13. Notes for the Next Stage

This notebook intentionally stays with the experimental single-task problem. The combined DFT/experimental dataset in the source folder can be used next for multitask GPR, where the task label distinguishes experimental hardness from the five semiempirical hardness models.